In [1]:
import numpy as np
import tensorflow as tf 
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,Dense

In [2]:
max_features = 10000
(X_train,y_train),(X_test,y_test)=imdb.load_data(num_words=max_features)
print(f'Training Data Shape :{X_train.shape},training labels shape:{y_train.shape}')
print(f'Testing Data Shape:{X_train.shape},Testing labels Shape:{y_test.shape}')


Training Data Shape :(25000,),training labels shape:(25000,)
Testing Data Shape:(25000,),Testing labels Shape:(25000,)


In [3]:
### inspecta sample review and its label
sample_review = X_train[0] 
sample_review

[1,
 14,
 22,
 16,
 43,
 530,
 973,
 1622,
 1385,
 65,
 458,
 4468,
 66,
 3941,
 4,
 173,
 36,
 256,
 5,
 25,
 100,
 43,
 838,
 112,
 50,
 670,
 2,
 9,
 35,
 480,
 284,
 5,
 150,
 4,
 172,
 112,
 167,
 2,
 336,
 385,
 39,
 4,
 172,
 4536,
 1111,
 17,
 546,
 38,
 13,
 447,
 4,
 192,
 50,
 16,
 6,
 147,
 2025,
 19,
 14,
 22,
 4,
 1920,
 4613,
 469,
 4,
 22,
 71,
 87,
 12,
 16,
 43,
 530,
 38,
 76,
 15,
 13,
 1247,
 4,
 22,
 17,
 515,
 17,
 12,
 16,
 626,
 18,
 2,
 5,
 62,
 386,
 12,
 8,
 316,
 8,
 106,
 5,
 4,
 2223,
 5244,
 16,
 480,
 66,
 3785,
 33,
 4,
 130,
 12,
 16,
 38,
 619,
 5,
 25,
 124,
 51,
 36,
 135,
 48,
 25,
 1415,
 33,
 6,
 22,
 12,
 215,
 28,
 77,
 52,
 5,
 14,
 407,
 16,
 82,
 2,
 8,
 4,
 107,
 117,
 5952,
 15,
 256,
 4,
 2,
 7,
 3766,
 5,
 723,
 36,
 71,
 43,
 530,
 476,
 26,
 400,
 317,
 46,
 7,
 4,
 2,
 1029,
 13,
 104,
 88,
 4,
 381,
 15,
 297,
 98,
 32,
 2071,
 56,
 26,
 141,
 6,
 194,
 7486,
 18,
 4,
 226,
 22,
 21,
 134,
 476,
 26,
 480,
 5,
 144,
 30,
 5535,
 18,

In [9]:
max_len = 500
X_train = sequence.pad_sequences(X_train,maxlen = max_len)
X_test = sequence.pad_sequences(X_test,maxlen=max_len)
X_train


array([[   0,    0,    0, ...,   19,  178,   32],
       [   0,    0,    0, ...,   16,  145,   95],
       [   0,    0,    0, ...,    7,  129,  113],
       ...,
       [   0,    0,    0, ...,    4, 3586,    2],
       [   0,    0,    0, ...,   12,    9,   23],
       [   0,    0,    0, ...,  204,  131,    9]])

In [12]:
## Train our simple RNN
model = Sequential()
model.add(Embedding(max_features,128,input_length=max_len)) ### Embedding layer
model.add(SimpleRNN(128,activation='relu'))
model.add(Dense(1,activation='sigmoid'))


In [13]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 500, 128)          1280000   
                                                                 
 simple_rnn (SimpleRNN)      (None, 128)               32896     
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1313025 (5.01 MB)
Trainable params: 1313025 (5.01 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [14]:
### Create an instance of early stopping callback
from tensorflow.keras.callbacks import EarlyStopping 
early_stopping =EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [15]:
early_stopping

In [23]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics='accuracy')

In [21]:
### Train the model with early stopping 
model.fit(X_train,y_train,epochs=10,batch_size=32,validation_split=0.2,callbacks=[early_stopping])


Epoch 1/10
625/625 [==============================] - 28s 45ms/step - loss: 766950.9375 - val_loss: 0.6074
Epoch 2/10
625/625 [==============================] - 31s 49ms/step - loss: 0.6786 - val_loss: 0.5735
Epoch 3/10
625/625 [==============================] - 30s 48ms/step - loss: 3.2784 - val_loss: 0.5462
Epoch 4/10
625/625 [==============================] - 29s 46ms/step - loss: 1.8976 - val_loss: 0.5260
Epoch 5/10
625/625 [==============================] - 30s 48ms/step - loss: 0.3836 - val_loss: 0.4885
Epoch 6/10
625/625 [==============================] - 28s 45ms/step - loss: 0.3100 - val_loss: 0.4489
Epoch 7/10
625/625 [==============================] - 29s 47ms/step - loss: 0.2350 - val_loss: 0.4309
Epoch 8/10
625/625 [==============================] - 29s 47ms/step - loss: 0.1707 - val_loss: 0.4361
Epoch 9/10
625/625 [==============================] - 29s 47ms/step - loss: 0.1322 - val_loss: 0.4489
Epoch 10/10
625/625 [==============================] - 29s 47ms/step - loss: 

In [24]:
### Saving the model file 
model.save('simple_rnn_imbd.h5')

c:\Users\My\Desktop\Study Material\Deep Learning project\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
